# Stroke Prediction Assignment - Version 2
Updated notebook with class imbalance fixes and zero_division handling.


In [1]:
import pandas as pd
import numpy as np
import os, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef


In [2]:
df=pd.read_csv('dataset.csv')
#if 'id' in df.columns: df=df.drop('id',axis=1)
if 'bmi' in df.columns: df['bmi']=df['bmi'].fillna(df['bmi'].median())#
for col in df.columns:
    if df[col].dtype=='object':
        enc=LabelEncoder()
        df[col]=enc.fit_transform(df[col].astype(str))
print(df.shape)
df.head()

(5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [4]:
print(df.dtypes)

id                     int64
gender                 int32
age                  float64
hypertension           int64
heart_disease          int64
ever_married           int32
work_type              int32
Residence_type         int32
avg_glucose_level    float64
bmi                  float64
smoking_status         int32
stroke                 int64
dtype: object


In [5]:
TARGET = "stroke"

X = df.drop(TARGET, axis=1)
y = df[TARGET]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
models={
'Logistic Regression':Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(max_iter=3000,class_weight='balanced'))]),
'Decision Tree':DecisionTreeClassifier(random_state=42,class_weight='balanced'),
'KNN':Pipeline([('scaler',StandardScaler()),('clf',KNeighborsClassifier())]),
'Naive Bayes':Pipeline([('scaler',StandardScaler()),('clf',GaussianNB())]),
'SVM':Pipeline([('scaler',StandardScaler()),('clf',SVC(probability=True,class_weight='balanced'))])
}
models.keys()

dict_keys(['Logistic Regression', 'Decision Tree', 'KNN', 'Naive Bayes', 'SVM'])

In [8]:
os.makedirs('model',exist_ok=True)
results=[]
for name,model in models.items():
    print(f'Training {name}')
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    try:
        auc=roc_auc_score(y_test,model.predict_proba(X_test)[:,1])
    except:
        auc=0
    results.append([name,accuracy_score(y_test,pred),auc,precision_score(y_test,pred,average='weighted',zero_division=0),recall_score(y_test,pred,average='weighted',zero_division=0),f1_score(y_test,pred,average='weighted',zero_division=0),matthews_corrcoef(y_test,pred)])
    joblib.dump(model,'model/'+name.lower().replace(' ','_')+'.pkl')
metric_df=pd.DataFrame(results,columns=['Model','Accuracy','AUC','Precision','Recall','F1','MCC'])
metric_df=metric_df.round(4)
metric_df.to_csv('model_metrics.csv',index=False)
metric_df

Training Logistic Regression
Training Decision Tree
Training KNN
Training Naive Bayes
Training SVM


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.7456,0.8364,0.9448,0.7456,0.8175,0.2597
1,Decision Tree,0.9335,0.5477,0.9187,0.9335,0.9255,0.1218
2,KNN,0.9501,0.6028,0.9217,0.9501,0.9286,0.0715
3,Naive Bayes,0.8669,0.8018,0.9283,0.8669,0.8933,0.2020
4,SVM,0.7613,0.7760,0.9347,0.7613,0.8276,0.2023


In [9]:
for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print("\n", name)
    print(pd.Series(pred).value_counts())


 Logistic Regression
0    732
1    290
Name: count, dtype: int64

 Decision Tree
0    992
1     30
Name: count, dtype: int64

 KNN
0    1019
1       3
Name: count, dtype: int64

 Naive Bayes
0    894
1    128
Name: count, dtype: int64

 SVM
0    764
1    258
Name: count, dtype: int64
